In [ ]:
from singleCAM_IROS._pipeline_support import _handle_dirpaths

from pathlib import Path

from tqdm import tqdm
from numpy.typing import NDArray
import numpy as np

from bloodmoon.io import simulation_files
from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.mask import count

import darksun as ds

ds.show.set_figures_darkbkg()

In [ ]:
def config_psfy_flag(dataset: str) -> bool:
    if dataset not in ['detected', 'reconstructed']:
        raise ValueError('Invalid dataset type.')
    flag: bool = (True if DATASET == 'reconstructed' else False)
    return flag

MASK_FITS: str = "wfm_mask_NTHT_20250725.fits"

SKYFIELD: str = "IROSDummy"
DATA_FITS: str = "iros_benchmark_2-50keV_mask_050_1040x17_1ks"

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "reconstructed"

UPS_X: int = 2
UPS_Y: int = 1

VIGNETTING: bool = True
PSFY: bool = config_psfy_flag(DATASET)

In [ ]:
# load filepaths
mask_path, simul_data, save_path = _handle_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
)
wfm: CodedMaskCamera = codedmask(mask_path, UPS_X, UPS_Y)
filepaths: dict[str, dict[str, Path]] = simulation_files(simul_data)

E_MIN, E_MAX = None, None

# data from camera A
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_MIN, E_max=E_MAX)
catalogueA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])

# data from camera B
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET])
catalogueB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

In [ ]:
from typing import Callable

from scipy.optimize import differential_evolution

from bloodmoon.coords import pos2shift
from bloodmoon.optim import model_sky, process_skyimg


def MSEloss(
    func: Callable[[float, float, float], NDArray],
) -> Callable[[tuple[float, float, float], NDArray, NDArray], float]:
    """
    Returns a loss function for source parameter optimization, given a routine for computing models.
    """
    def f(args: tuple[float, float, float], true: NDArray, sigma: NDArray) -> float:
        """Compute the Mean Square Error loss between source model and true data."""
        modeled = func(*args)
        mse = np.mean(np.square((true - modeled) / sigma))
        return float(mse)
    
    return f


def config_optimiser(
    camera: CodedMaskCamera,
    vignetting: bool = True,
    psfy: bool = True,
    camera_coding_power: float = 0.85,
) -> Callable[[NDArray, tuple[int, int], bool], tuple[float, ...]]:
    """
    Configures the IROS optimiser for source parameters fitting.
    """
    def _ModelShiftFluence(
        pos: tuple[int, int],
    ) -> Callable[[float, float, float], NDArray]:
        """
        Initialises the source model.
        """
        def f(sx: float, sy: float, cts: float) -> NDArray:
            """Models the source sky image."""
            modeled = model_sky(camera, sx, sy, cts, vignetting, psfy)
            return process_skyimg(camera, modeled, pos)
        
        return f

    def optimise(
        sky: NDArray,
        arg_sky: tuple[int, int],
        verbose: bool = False,
    ) -> tuple[float, ...]:
        """
        Performs the optimization to fit a point source model to sky image data.
        """
        model_shift_flux = _ModelShiftFluence(arg_sky)
        loss = MSEloss(model_shift_flux)

        # setup `curve_fit` params
        # - setup source data + std
        sky_ydata = process_skyimg(camera, sky, arg_sky)
        _sky_ydata_sigma = np.sqrt(np.clip(sky_ydata, a_min=0.0, a_max=None))
        sky_ydata_sigma = _sky_ydata_sigma.max() - _sky_ydata_sigma + 1
        # - extract source coords and counts starting values
        sx_start, sy_start = pos2shift(camera, *arg_sky)
        sky_peak = sky[*arg_sky]
        cts_start = (
            sky_peak / camera_coding_power if psfy else sky_peak
        )
        start_params_vals = (sx_start, sy_start, cts_start)
        # - setup fit params boundaries
        #    - the shifts are allowed to fluctuate in a small pixel box since
        #      the extracted position is close enough to the true source pos
        #      A small box also account for superimposed or close sources,
        #      which may introduce biases in the source fit procedure
        #    - the box is built from the digital upsampling since in the worst
        #      case (source at 45 deg wrt optical axis) the high energy detected
        #      photons median absorption distance is 225um. The projection of this
        #      distance on the camera plane is always smaller than `px_size / ups`  
        #    - the fluence cannot be smaller than the one observed at the peak,
        #      and we insert a lower value just for precaution (if simulating
        #      for example an infinite detector spatial resolution)
        px_dim_x, px_dim_y = (
            camera.specs.mask_deltax / camera.upscale_f.x,
            camera.specs.mask_deltay / camera.upscale_f.y,
        )
        dx, dy = camera.upscale_f.x, camera.upscale_f.y
        bounds = [
            (
                max(sx_start - dx * px_dim_x, camera.bins_sky.x[0]),
                max(sy_start - dy * px_dim_y, camera.bins_sky.y[0]),
                0.95 * sky_peak,
            ),
            (
                min(sx_start + dx * px_dim_x, camera.bins_sky.x[-1]),
                min(sy_start + dy * px_dim_y, camera.bins_sky.y[-1]),
                1.25 * sky_peak,
            ),
        ]
        # - setup differential evolution kwargs
        diff_evl = {
            'strategy': ...,
            'tol': ...,
            'workers': 1,
            'x0': start_params_vals,
        }

        # perform optimisation
        results = differential_evolution(
            func=loss,
            bounds=bounds,
            args=(sky_ydata, sky_ydata_sigma),
            **diff_evl,
        )
        sx, sy, cts = map(float, results.x)
        
        if verbose:
            print(
                f'\n'
                f'## Optimisation Results:\n'
                f'  - fluence START: {cts_start}\n'
                f'  - shifts START (x, y): {sx_start}, {sy_start}\n'

                f'  - fluence OPTIM.: {cts}\n'
                f'  - shifts OPTIM. (x, y): {sx}, {sy}\n'

                f'  - fluence GAIN %: {(cts - cts_start) * 100 / cts_start:.3f}\n'
                f'  - shift_x GAIN %: {np.sign(sx_start) * (sx - sx_start) * 100 / sx_start:.3f}\n'
                f'  - shift_y GAIN %: {np.sign(sy_start) * (sy - sy_start) * 100 / sy_start:.3f}'
            )
            print(
                f'## Fit Report:\n'
                f'  - optimisation successful: {results.success}\n'
                f'  - loss sqrt at solution: {np.sqrt(results.fun)}\n'
                f'  - num of iters: {results.nit} (loss calls: {results.nfev})\n'

                f'  - procedure msg: {results.message}\n'
                f'  - success status {results.status}\n'
            )

        return sx, sy, cts
    
    return optimise

NameError: name 'CodedMaskCamera' is not defined